In [15]:
from datetime import datetime
import os
!pip install pymupdf
!pip install pdf2image
!pip instal fitz
import os
import fitz
from pathlib import Path
from data.utils import get_project_root

ROOT_DIR = get_project_root()
pdf_dirs = os.path.join(ROOT_DIR, "data", "fake_data", "urssaf_vigilance")
versions = [d for d in Path(pdf_dirs).iterdir() if d.is_dir() and d.name.startswith("2026")]
if not versions:
    raise FileNotFoundError("Aucun dossier de version trouvé.")
pdf_dir = sorted(versions)[-1]
img_dir = Path(ROOT_DIR) / "data" / "fake_data" / "images_clean"
img_dir.mkdir(parents=True, exist_ok=True)

print(f"Traitement avec PyMuPDF dans : {pdf_dir}")

for pdf_path in pdf_dir.glob("*.pdf"):
    doc = fitz.open(pdf_path)

    for i, page in enumerate(doc):
        pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))

        save_path = img_dir / f"{pdf_path.stem}_{i}.png"
        pix.save(str(save_path))
        print(f"Exporté : {save_path.name}")

    doc.close()

  Obtaining dependency information for pymupdf from https://files.pythonhosted.org/packages/f5/fb/a3f1f8813f6e93c65d1f7ebca6530a889f1ae109229b537f7a617b2aab57/pymupdf-1.27.2-cp310-abi3-win_amd64.whl.metadata
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   ---------------------------------------- 0.1/19.2 MB 3.6 MB/s eta 0:00:06
   - -------------------------------------- 0.7/19.2 MB 8.3 MB/s eta 0:00:03
   --- ------------------------------------ 1.7/19.2 MB 13.7 MB/s eta 0:00:02
   ------ --------------------------------- 3.3/19.2 MB 19.0 MB/s eta 0:00:01
   --------- ------------------------------ 4.5/19.2 MB 20.5 MB/s eta 0:00:01
   -------------- ------------------------- 6.8/19.2 MB 25.5 MB/s eta 0:00:01
   ----------------- ---------------------- 8.5/19.2 MB 27.2 MB/s eta 0:00:01
   ---------------------- ----------------- 11.0/19.2 MB 38.5 MB/s eta 0:00:01
   --------------------------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: unknown command "instal" - maybe you meant "install"



Traitement avec PyMuPDF dans : C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\urssaf_vigilance\20260317_172717
Exporté : urssaf_218901031_030_0.png
Exporté : urssaf_218901072_031_0.png
Exporté : urssaf_218901825_000_0.png
Exporté : urssaf_218901866_001_0.png
Exporté : urssaf_218901973_002_0.png
Exporté : urssaf_218902088_003_0.png
Exporté : urssaf_218902112_004_0.png
Exporté : urssaf_218902179_005_0.png
Exporté : urssaf_218902252_006_0.png
Exporté : urssaf_218902260_007_0.png
Exporté : urssaf_218902260_008_0.png
Exporté : urssaf_218902286_009_0.png
Exporté : urssaf_218902609_017_0.png
Exporté : urssaf_218902948_032_0.png


In [17]:
import cv2
import numpy as np
import random
from pathlib import Path

input_dir = Path(img_dir)
output_dir = Path(os.path.join(ROOT_DIR, "data", "fake_data","images_noisy"))
output_dir.mkdir(exist_ok=True)

def add_noise(image):
    noise = np.random.normal(0, 25, image.shape).astype(np.uint8)
    return cv2.add(image, noise)

def add_blur(image):
    k = random.choice([3,5])
    return cv2.GaussianBlur(image, (k,k), 0)

def add_motion_blur(image):
    size = random.randint(5, 15)
    kernel = np.zeros((size, size))
    kernel[int((size-1)/2), :] = np.ones(size)
    kernel = kernel / size
    return cv2.filter2D(image, -1, kernel)

def rotate(image):
    angle = random.uniform(-5, 5)
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1)
    return cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REPLICATE)

def perspective_transform(image):
    h, w = image.shape[:2]

    shift = 20
    pts1 = np.float32([[0,0],[w,0],[0,h],[w,h]])
    pts2 = np.float32([
        [random.randint(0,shift), random.randint(0,shift)],
        [w-random.randint(0,shift), random.randint(0,shift)],
        [random.randint(0,shift), h-random.randint(0,shift)],
        [w-random.randint(0,shift), h-random.randint(0,shift)]
    ])

    M = cv2.getPerspectiveTransform(pts1, pts2)
    return cv2.warpPerspective(image, M, (w, h))

def jpeg_compression(image):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(30, 70)]
    _, encimg = cv2.imencode('.jpg', image, encode_param)
    return cv2.imdecode(encimg, 1)

def change_brightness(image):
    factor = random.uniform(0.6, 1.4)
    return np.clip(image * factor, 0, 255).astype(np.uint8)

def degrade(image):
    if random.random() < 0.7:
        image = rotate(image)

    if random.random() < 0.5:
        image = perspective_transform(image)

    if random.random() < 0.6:
        image = add_blur(image)

    if random.random() < 0.5:
        image = add_motion_blur(image)

    if random.random() < 0.7:
        image = add_noise(image)

    if random.random() < 0.8:
        image = change_brightness(image)

    if random.random() < 0.9:
        image = jpeg_compression(image)

    return image

In [18]:
for img_path in input_dir.glob("*.png"):
    image = cv2.imread(str(img_path))

    for i in range(3):  # 3 versions différentes
        degraded = degrade(image)
        output_path = output_dir / f"{img_path.stem}_noisy_{i}.jpg"
        print(output_path)
        cv2.imwrite(str(output_path), degraded)

C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901031_030_0_noisy_0.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901031_030_0_noisy_1.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901031_030_0_noisy_2.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901072_031_0_noisy_0.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901072_031_0_noisy_1.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901072_031_0_noisy_2.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901825_000_0_noisy_0.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\urssaf_218901825_000_0_noisy_1.jpg
C:\Users\seerm\PycharmProjects\hackathon_ipssi_2026\data\fake_data\images_noisy\